***Loading Datasets***

In [1]:
import pandas as pd

df = pd.read_csv('2021.csv')
df.drop('Unnamed: 1',axis=1,inplace=True)       # removing index column
df.rename(columns={'Unnamed: 0':'Stage'},inplace=True)  # Required for prediction
df_c1 = df      # for Prediction
df1 = df.iloc[:16].reset_index(drop=True).fillna('')#opening stages of that year
df2 = df.iloc[16:32].reset_index(drop=True).fillna('')#elimination stage 16 rows
df3 = df.iloc[32:].reset_index(drop=True).fillna('')#playoff stage 
df_21 = pd.concat([df1,df2,df3],keys=['Opening Stage','Elimination Stage','Playoff Stage']).drop('Stage',axis=1)  # for other features... removing Stage column
#print("df1:\n",df_21)

df = pd.read_csv('2022.csv')
df.drop('Unnamed: 1',axis=1,inplace=True)
df.rename(columns={'Unnamed: 0':'Stage'},inplace=True)
df_c2 = df
df1 = df.iloc[:16].reset_index(drop=True).fillna('')
df2 = df.iloc[16:32].reset_index(drop=True).fillna('')
df3 = df.iloc[32:].reset_index(drop=True).fillna('')
df_22 = pd.concat([df1,df2,df3],keys=['Opening Stage','Elimination Stage','Playoff Stage']).drop('Stage',axis=1)

df = pd.read_csv('2023.csv')
df.drop('Unnamed: 1',axis=1,inplace=True)
df.rename(columns={'Unnamed: 0':'Stage'},inplace=True)
df_c3 = df
df1 = df.iloc[:16].reset_index(drop=True).fillna('')
df2 = df.iloc[16:32].reset_index(drop=True).fillna('')
df3 = df.iloc[32:].reset_index(drop=True).fillna('')
df_23 = pd.concat([df1,df2,df3],keys=['Opening Stage','Elimination Stage','Playoff Stage']).drop('Stage',axis=1)

df = pd.read_csv('2024.csv')
df.drop('Unnamed: 1',axis=1,inplace=True)
df.rename(columns={'Unnamed: 0':'Stage'},inplace=True)
df_c4 = df
df1 = df.iloc[:16].reset_index(drop=True).fillna('')
df2 = df.iloc[16:32].reset_index(drop=True).fillna('')
df3 = df.iloc[32:].reset_index(drop=True).fillna('')
df_24 = pd.concat([df1,df2,df3],keys=['Opening Stage','Elimination Stage','Playoff Stage']).drop('Stage',axis=1)


df = pd.concat([df_c1,df_c2,df_c3,df_c4]).reset_index(drop=True).fillna("")
df

,Stage,Team,Matches,Rounds,RD,Round 1,Round 2,Round 3,Round 4,Round 5
0,Opening Stage,FaZe Clan,3-0,64-43,21,Team Spirit,ENCE,Virtus.pro,,
1,Opening Stage,Copenhagen Flames,3-0,75-52,23,Astralis,BIG,HEROIC,,
2,Opening Stage,ENCE,3-1,86-71,15,GODSENT,FaZe Clan,MOUZ,BIG,
3,Opening Stage,Entropiq,3-1,87-68,19,BIG,Astralis,Movistar Riders,HEROIC,
4,Opening Stage,Virtus.pro,3-1,99-96,3,paiN Gaming,Movistar Riders,FaZe Clan,Team Spirit,
...,...,...,...,...,...,...,...,...,...,...
155,Playoff Stage,G2 Esports,1-1,63-51,12,MOUZ,Natus Vincere,,,
156,Playoff Stage,Team Spirit,0-1,40-45,-5,FaZe Clan,,,,
157,Playoff Stage,Cloud9,0-1,10-26,-16,Team Vitality,,,,
158,Playoff Stage,Eternal Fire,0-1,22-29,-7,Natus Vincere,,,,


In [92]:
import pandas as pd
import plotly.express as px


round_columns = ['Round 1', 'Round 2', 'Round 3', 'Round 4', 'Round 5']

team_counts = df[round_columns].apply(lambda x: x != '').sum()


team_counts_df = pd.DataFrame(team_counts, columns=['Team Count']).reset_index()
team_counts_df.columns = ['Round', 'Team Count']

fig = px.bar(
    team_counts_df,
    x='Round',
    y='Team Count',
    title="Number of Teams in Each Round (All Stages)",
    labels={"Team Count": "Number of Teams", "Round": "Round"},
    template="plotly_dark"
)

fig.show()


***Prediction***

In [93]:
def calculate_avg_metrics(group):
    total_wins = group['Matches'].apply(lambda x: int(x.split('-')[0])).sum()
    total_losses = group['Matches'].apply(lambda x: int(x.split('-')[1])).sum()
    total_matches = total_wins + total_losses
    avg_win_rate = total_wins / total_matches
    avg_rd = group['RD'].mean()
    stages = group['Stage'].tolist()
    playoff_appearances = stages.count('Playoff Stage')
    return pd.Series({
        'Avg Win Rate': avg_win_rate,
        'Avg RD': avg_rd,
        'Playoff Appearances': playoff_appearances
    })

# Group by 'Team' and apply the function
team_stats = df.groupby('Team', group_keys=False).apply(calculate_avg_metrics).reset_index()

# Show the resulting team_stats DataFrame
team_stats


,Team,Avg Win Rate,Avg RD,Playoff Appearances
0,9INE,0.000000,-24.000000,0.0
1,9z Team,0.000000,-21.000000,0.0
2,AMKAL ESPORTS,0.000000,-17.000000,0.0
3,Apeks,0.533333,-1.500000,1.0
4,Astralis,0.428571,-4.666667,0.0
5,BIG,0.333333,-10.000000,0.0
6,Bad News Eagles,0.333333,-8.666667,0.0
7,CPH Flames,0.000000,-10.000000,1.0
8,Cloud9,0.583333,1.500000,1.0
9,Complexity Gaming,0.307692,-30.000000,0.0


In [94]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create subplots with 1 row and 3 columns
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=["Avg Win Rate", "Avg RD", "Playoff Appearances"],
    horizontal_spacing=0.1  # Adjust spacing between subplots
)

# Add Avg Win Rate
fig.add_trace(
    go.Bar(
        x=team_stats['Avg Win Rate'],
        y=team_stats['Team'],
        orientation='h',
        name='Avg Win Rate',
        marker=dict(color='lightblue')
    ),
    row=1, col=1
)

# Add Avg RD
fig.add_trace(
    go.Bar(
        x=team_stats['Avg RD'],
        y=team_stats['Team'],
        orientation='h',
        name='Avg RD',
        marker=dict(color='lightgreen')
    ),
    row=1, col=2
)

# Add Playoff Appearances
fig.add_trace(
    go.Bar(
        x=team_stats['Playoff Appearances'],
        y=team_stats['Team'],
        orientation='h',
        name='Playoff Appearances',
        marker=dict(color='lightcoral')
    ),
    row=2, col=1
)

# Update layout for black background and set axis properties
fig.update_layout(
    title='Team Performance Metrics',
    plot_bgcolor='black',
    paper_bgcolor='black',
    font=dict(color='white'),
    showlegend=False,  # Disable legend since titles are self-explanatory
)

# Update individual subplot styles
for i in range(1, 4):
    fig['layout'][f'xaxis{i}'].update(showgrid=True)
    fig['layout'][f'yaxis{i}'].update(showgrid=True)

# Display the plot
fig.show()


In [95]:

fig = px.scatter(
    team_stats,
    x="Avg Win Rate",
    y="Avg RD",
    size="Playoff Appearances",
    color="Team",
    hover_name="Team",
    title="Team Performance Bubble Chart",
    labels={"Avg_Win_Rate":"Average Win Rate","Avg_RD":"Average RD"},
    template="plotly_dark",
    size_max=30
)

#Customize layout
fig.update_layout(
    plot_bgcolor="grey",
    #paper_bgcolor="black",
    font=dict(color="white"),
    title_font=dict(size=20),
)


fig.show()

In [96]:
def predict_outcome(team_stats, playoff_threshold=2, win_rate_threshold=0.4):
    likely_qualifiers = team_stats[
        (team_stats['Playoff Appearances'] >= playoff_threshold) &
        (team_stats['Avg Win Rate'] >= win_rate_threshold)
    ]
    
    likely_winners = likely_qualifiers.sort_values(by=['Avg RD', 'Avg Win Rate'], ascending=False).head(3)
    print(likely_winners)
    return likely_qualifiers['Team'], likely_winners['Team']

qualifiers, winners = predict_outcome(team_stats)

print("Teams likely to qualify:", qualifiers.tolist())
print("Teams likely to win:", winners.tolist())


             Team  Avg Win Rate     Avg RD  Playoff Appearances
21     G2 Esports      0.642857  15.375000                  2.0
48  Team Vitality      0.692308  11.875000                  3.0
38  Natus Vincere      0.760000  11.428571                  3.0
Teams likely to qualify: ['FURIA Esports', 'FaZe Clan', 'G2 Esports', 'Heroic', 'Natus Vincere', 'Ninjas in Pyjamas', 'Team Spirit', 'Team Vitality']
Teams likely to win: ['G2 Esports', 'Team Vitality', 'Natus Vincere']


In [97]:
import plotly.express as px



# Scatter Plot
fig = px.scatter(
    team_stats,
    x="Avg Win Rate",
    y="Avg RD",
    size="Playoff Appearances",
    color="Team",
    #hover_name="Team",
    title="Predicted Winners and Qualifiers",
    labels={"Avg Win Rate": "Average Win Rate", "Avg RD": "Average RD"},
    template="plotly_dark"  
)

fig.show()


In [98]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Call the predict_outcome function to get qualifiers and winners
qualifiers, winners = predict_outcome(team_stats)

# Filter the original data for the top 3 likely winners
top_teams = team_stats[team_stats['Team'].isin(winners)]

# Create subplots
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Average Win Rate", "Average RD", "Playoff Appearances"),
    horizontal_spacing=0.1
)

# Add Avg Win Rate bar chart (vertical)
fig.add_trace(
    go.Bar(
        x=top_teams["Team"],
        y=top_teams["Avg Win Rate"],
        name="Avg Win Rate",
        marker=dict(color="lightblue")
    ),
    row=1, col=1
)

# Add Avg RD bar chart (vertical)
fig.add_trace(
    go.Bar(
        x=top_teams["Team"],
        y=top_teams["Avg RD"],
        name="Avg RD",
        marker=dict(color="lightgreen")
    ),
    row=1, col=2
)

# Add Playoff Appearances bar chart (vertical)
fig.add_trace(
    go.Bar(
        x=top_teams["Team"],
        y=top_teams["Playoff Appearances"],
        name="Playoff Appearances",
        marker=dict(color="lightcoral")
    ),
    row=1, col=3
)

# Update layout
fig.update_layout(
    title="Top 3 Predicted Winners - Performance Metrics",
    plot_bgcolor="black",
    paper_bgcolor="black",
    font=dict(color="white"),
    height=600,
    width=1200,
)

# Display the plot
fig.show()


             Team  Avg Win Rate     Avg RD  Playoff Appearances
21     G2 Esports      0.642857  15.375000                  2.0
48  Team Vitality      0.692308  11.875000                  3.0
38  Natus Vincere      0.760000  11.428571                  3.0


***Win Rate***

In [4]:
def calculate_win_rate(row):
    wins = int(row['Matches'].split('-')[0])
    losses = int(row['Matches'].split('-')[1])
    total_matches = wins + losses
    win_rate = wins / total_matches
    return win_rate

# df_24['Win Rate'] = df_24.apply(calculate_win_rate, axis=1)
# df_24[['Team', 'Win Rate']]

df_24['Win Rate'] = df_24.apply(calculate_win_rate, axis=1)
# df_24['Win Rate'] = df_24['Win Rate'] * 100
# df_24['Win Rate'] = df_24['Win Rate'].apply(lambda x: f"{x:.0f}%")
df_24[['Team', 'Win Rate']]

Team  Win Rate
Opening Stage     0               HEROIC  1.000000
                  1               Cloud9  1.000000
                  2         Eternal Fire  0.750000
                  3             ECSTATIC  0.750000
                  4          paiN Gaming  0.750000
                  5     Imperial Esports  0.600000
                  6          The MongolZ  0.600000
                  7        FURIA Esports  0.600000
                  8                  SAW  0.400000
                  9               Legacy  0.400000
                  10         GamerLegion  0.400000
                  11  Lynn Vision Gaming  0.250000
                  12                ENCE  0.250000
                  13               Apeks  0.250000
                  14       AMKAL ESPORTS  0.000000
                  15                 KOI  0.000000
Elimination Stage 0          Team Spirit  1.000000
                  1                 MOUZ  1.000000
                  2         Eternal Fire  0.750000
                  3               Cloud9  0.750000
                  4        Team Vitality  0.750000
                  5        Natus Vincere  0.600000
                  6           G2 Esports  0.600000
                  7            FaZe Clan  0.600000
                  8    Complexity Gaming  0.400000
                  9           Virtus.pro  0.400000
                  10         paiN Gaming  0.400000
                  11    Imperial Esports  0.250000
                  12            ECSTATIC  0.250000
                  13              HEROIC  0.250000
                  14         The MongolZ  0.000000
                  15       FURIA Esports  0.000000
Playoff Stage     0        Natus Vincere  1.000000
                  1            FaZe Clan  0.666667
                  2        Team Vitality  0.500000
                  3           G2 Esports  0.500000
                  4          Team Spirit  0.000000
                  5               Cloud9  0.000000
                  6         Eternal Fire  0.000000
                  7                 MOUZ  0.000000

In [9]:
import plotly.express as px

fig_win_rate = px.bar(df_24, 
                      x='Win Rate', 
                      y='Team', 
                      title='Win Rate Per Stage', 
                      labels={'Win Rate': 'Win Rate', 'Team': 'Team' , 'Stage' : df['Stage']}, 
                      color='Win Rate', 
                      color_continuous_scale='Viridis', 
                      orientation='h', 
                      template="plotly_dark")
fig_win_rate.show()

***Round Difference (RD) Analysis***

In [101]:
# Win Rate Required
correlation = df_24['RD'].corr(df_24['Win Rate'])
print(f"Correlation between RD and Win Rate: {correlation:.2f}")

Correlation between RD and Win Rate: 0.79


In [102]:
import plotly.express as px
import pandas as pd

fig = px.scatter(
    df_24, 
    x='RD', 
    y='Win Rate', 
    title='Correlation between Round Difference (RD) and Win Rate',
    labels={'RD': 'Round Difference (RD)', 'Win Rate': 'Win Rate'},
    trendline="ols",
    template="plotly_dark"
)

fig.show()


***Average Rounds per Game***

In [103]:
def total_matches(row):
    wins = int(row['Matches'].split('-')[0])
    losses = int(row['Matches'].split('-')[1])
    return wins + losses

df_24['Total Matches'] = df_24.apply(total_matches, axis=1)
df_24['Rounds Won'] = df_24['Rounds'].str.split('-').str[0].astype(int)
df_24['Rounds Lost'] = df_24['Rounds'].str.split('-').str[1].astype(int)

df_24['Avg Rounds Won'] = (df_24['Rounds Won'] / df_24['Total Matches']).astype(int)
df_24['Avg Rounds Lost'] = (df_24['Rounds Lost'] / df_24['Total Matches']).astype(int)

df_24[['Team', 'Avg Rounds Won', 'Avg Rounds Lost']]



Team  Avg Rounds Won  Avg Rounds Lost
Opening Stage     0               HEROIC              19               13
                  1               Cloud9              19               15
                  2         Eternal Fire              19               15
                  3             ECSTATIC              18               14
                  4          paiN Gaming              13               10
                  5     Imperial Esports              22               21
                  6          The MongolZ              20               16
                  7        FURIA Esports              20               15
                  8                  SAW              16               19
                  9               Legacy              14               15
                  10         GamerLegion              17               18
                  11  Lynn Vision Gaming              10               15
                  12                ENCE              13               18
                  13               Apeks              11               16
                  14       AMKAL ESPORTS              15               21
                  15                 KOI              10               18
Elimination Stage 0          Team Spirit              21               13
                  1                 MOUZ              17                8
                  2         Eternal Fire              17               12
                  3               Cloud9              18               12
                  4        Team Vitality              18               17
                  5        Natus Vincere              21               23
                  6           G2 Esports              22               18
                  7            FaZe Clan              16               14
                  8    Complexity Gaming              17               23
                  9           Virtus.pro              18               20
                  10         paiN Gaming              21               21
                  11    Imperial Esports              13               16
                  12            ECSTATIC              20               23
                  13              HEROIC              12               17
                  14         The MongolZ              20               23
                  15       FURIA Esports              13               22
Playoff Stage     0        Natus Vincere              31               28
                  1            FaZe Clan              33               32
                  2        Team Vitality              27               20
                  3           G2 Esports              31               25
                  4          Team Spirit              40               45
                  5               Cloud9              10               26
                  6         Eternal Fire              22               29
                  7                 MOUZ              15               26

In [104]:
import plotly.express as px

# Bar chart for Avg Rounds Won and Avg Rounds Lost
fig = px.bar(
    df_24,
    x="Team",
    y=["Avg Rounds Won", "Avg Rounds Lost"],
    title="Average Rounds Won and Lost per Team",
    labels={"value": "Average Rounds", "variable": "Metric"},
    barmode="group",
    template="plotly_dark" 
)

fig.show()



***Ranking Teams by Round Efficiency***

In [105]:
df_24['Total Rounds Played'] = df_24['Rounds Won'] + df_24['Rounds Lost']
df_24['Round Efficiency'] = df_24['Rounds Won'] / df_24['Total Rounds Played']

df_24[['Team', 'Round Efficiency']]

Team  Round Efficiency
Opening Stage     0               HEROIC          0.591837
                  1               Cloud9          0.557692
                  2         Eternal Fire          0.550000
                  3             ECSTATIC          0.558140
                  4          paiN Gaming          0.568421
                  5     Imperial Esports          0.518349
                  6          The MongolZ          0.560440
                  7        FURIA Esports          0.571429
                  8                  SAW          0.461111
                  9               Legacy          0.473333
                  10         GamerLegion          0.486188
                  11  Lynn Vision Gaming          0.396040
                  12                ENCE          0.429688
                  13               Apeks          0.409091
                  14       AMKAL ESPORTS          0.422018
                  15                 KOI          0.360465
Elimination Stage 0          Team Spirit          0.613208
                  1                 MOUZ          0.675325
                  2         Eternal Fire          0.579832
                  3               Cloud9          0.598361
                  4        Team Vitality          0.517241
                  5        Natus Vincere          0.475771
                  6           G2 Esports          0.550000
                  7            FaZe Clan          0.531646
                  8    Complexity Gaming          0.423645
                  9           Virtus.pro          0.476440
                  10         paiN Gaming          0.495370
                  11    Imperial Esports          0.445378
                  12            ECSTATIC          0.468571
                  13              HEROIC          0.428571
                  14         The MongolZ          0.465649
                  15       FURIA Esports          0.377358
Playoff Stage     0        Natus Vincere          0.525424
                  1            FaZe Clan          0.512690
                  2        Team Vitality          0.568421
                  3           G2 Esports          0.552632
                  4          Team Spirit          0.470588
                  5               Cloud9          0.277778
                  6         Eternal Fire          0.431373
                  7                 MOUZ          0.365854

In [109]:
import plotly.express as px


fig = px.bar(ranked_teams,
             x='Round Efficiency',
             y='Team',
             title='Teams Ranked by Round Efficiency',
             labels={'Round Efficiency': 'Round Efficiency', 'Team': 'Teams'},
             color='Round Efficiency', 
             color_continuous_scale='Viridis',
             orientation='h',
             template="plotly_dark"
            )

# Show the plot
fig.show()


***Identify Top 3 Teams by Win Rate***

In [107]:
top_3_teams = df_24.sort_values(by='Win Rate', ascending=False).head(3).reset_index()
print(top_3_teams[['Team', 'Win Rate']])


            Team  Win Rate
0         HEROIC       1.0
1    Team Spirit       1.0
2  Natus Vincere       1.0
